# Alignment Integration

Temporal alignment between source speech timing and target-language TTS audio.
This is the hard problem: a 3-second English phrase might take 5 seconds in Spanish.

Covers segment metrics, fallback policy, and global timeline optimization.

## Setup

In [ ]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

IMAGES_DIR = Path.cwd() / "images"
IMAGES_DIR.mkdir(exist_ok=True)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"IMAGES_DIR:   {IMAGES_DIR}")

## Load Cached Transcripts

Load from `pipeline_data` (no API call needed).

In [ ]:
en_dir = PROJECT_ROOT / "pipeline_data" / "api" / "transcriptions" / "whisper"
es_dir = PROJECT_ROOT / "pipeline_data" / "api" / "translations" / "argos"

en_files = sorted(en_dir.glob("*.json"))
es_files = sorted(es_dir.glob("*.json"))

assert en_files, f"No EN transcripts found in {en_dir}"
assert es_files, f"No ES translations found in {es_dir}"

en_path = en_files[0]
es_path = es_files[0]

with open(en_path) as f:
    en_transcript = json.load(f)
with open(es_path) as f:
    es_transcript = json.load(f)

print(f"EN transcript: {en_path.name}  ({len(en_transcript.get('segments', []))} segments)")
print(f"ES transcript: {es_path.name}  ({len(es_transcript.get('segments', []))} segments)")

## Segment Timing Metrics

Compute predicted stretch factor and overflow for each segment.

The syllable-based duration heuristic estimates TTS output length at ~4.5 syllables/second
for Romance languages (~15 chars/s). For each segment we compare the predicted TTS duration
against the source-language time window to get `predicted_stretch` (1.0 = perfect fit,
1.3 = 30% too long).

**Notice what happens:** many segments have stretch factors well above 1.0. The translator
doesn't know about the timing budget — it just rewrites text without considering how long
it takes to speak. This is the core problem you'll address in the tasks below.

In [ ]:
from foreign_whispers import (
    AlignAction, AlignedSegment, SegmentMetrics,
    compute_segment_metrics, decide_action,
)

all_metrics = compute_segment_metrics(en_transcript, es_transcript)
bad = [m for m in all_metrics if m.predicted_stretch > 1.5]

print(f"Total segments : {len(all_metrics)}")
print(f"Stretch > 1.5x : {len(bad)}  ({100*len(bad)/max(len(all_metrics),1):.0f}%)")
print("\nWorst 5:")
for m in sorted(bad, key=lambda x: -x.predicted_stretch)[:5]:
    print(f"  seg {m.index:3d}  stretch={m.predicted_stretch:.2f}x  overflow={m.overflow_s:.1f}s")
    print(f"    EN: {m.source_text[:55]}")
    print(f"    ES: {m.translated_text[:55]}")

## Visualize Stretch Distribution

In [ ]:
# Stretch distribution — show BOTH the legacy baseline predictor
# (syllables/4.5) and the fitted predictor that ships in alignment.py
# today. The "before" curve documents the problem Task 1 set out to fix;
# the "after" curve is what every cell below this one operates on.

import matplotlib.pyplot as plt
import numpy as np
from foreign_whispers.alignment import _estimate_duration_baseline

improved_stretches = [m.predicted_stretch for m in all_metrics]
baseline_stretches = [
    _estimate_duration_baseline(m.translated_text) / m.source_duration_s
    for m in all_metrics
]

upper = max(max(baseline_stretches, default=1.0), max(improved_stretches, default=1.0)) * 1.05
bins  = np.linspace(0, upper, 30)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(baseline_stretches, bins=bins, alpha=0.55, color="#888",
        label=f"baseline (syllables/4.5)  mean={np.mean(baseline_stretches):.2f}")
ax.hist(improved_stretches, bins=bins, alpha=0.55, color="#5099d8",
        label=f"fitted predictor          mean={np.mean(improved_stretches):.2f}")
ax.axvline(x=1.0, color="black", linestyle="--", linewidth=0.8, label="perfect fit (stretch = 1)")
ax.set_xlabel("Predicted stretch factor (predicted_tts_s / source_window_s)")
ax.set_ylabel("Segment count")
ax.set_title("Stretch Distribution — baseline vs fitted predictor")
ax.legend()
fig.tight_layout()
fig.savefig(IMAGES_DIR / "stretch_distribution.png", dpi=150)
plt.show()
print(f"Saved to {IMAGES_DIR / 'stretch_distribution.png'}")


---

## Task 1: Improve TTS Duration Prediction

The stretch factors above rely on a crude heuristic: ~15 characters/second for Spanish. Look at the worst segments — the heuristic is often wrong because character count ignores syllable structure, pauses, and speaking rate.

**Goal:** Replace the heuristic with a better duration predictor and measure whether it reduces alignment errors.

**Approach:**
1. Collect ground-truth durations by running TTS on a sample of segments and measuring actual WAV duration
2. Compare predictors: character count, syllable count (use a Spanish syllabifier), and a simple regression model trained on (text features → actual TTS duration)
3. Plug your predictor into `compute_segment_metrics` by modifying the `predicted_tts_duration_s` calculation in `foreign_whispers/alignment.py`

**File to modify:** `foreign_whispers/alignment.py` — the `_estimate_duration` helper

**Evaluation:**
- Mean absolute duration error (predicted vs actual TTS output)
- Calibration: does the predictor work equally well for short and long utterances?
- Downstream: does the improved predictor change the action distribution (fewer `REQUEST_SHORTER` or `FAIL`)? Re-run the policy histogram below to check.

### Task 1 — Improve TTS duration prediction

**Approach chosen.** Closed-form linear regression on three cheap text
features — character count, syllable count, and word count — with
coefficients fitted offline on every `(text, raw_duration_s)` pair from
this project's `.align.json` sidecars and **baked into the source as
constants**. No model file to load, no runtime cost.

**Why.** The single-feature `syllables / 4.5` heuristic ignores
character density and word boundaries — two segments with the same
syllable count can have very different real TTS durations. A
4-coefficient model captures the joint contribution of those features
while staying small enough to fit reliably on ~40 training samples.

**Changes in `foreign_whispers/alignment.py`**:

- New constants `_DUR_COEF_CHARS / _DUR_COEF_SYLL / _DUR_COEF_WORDS / _DUR_BIAS`.
- `_estimate_duration()` rewritten to evaluate the closed-form regression with a min-syllable floor.
- `_estimate_duration_baseline()` exposes the legacy `syllables / 4.5` formula so this notebook can A/B compare it against the fitted predictor without inlining the formula.


In [ ]:
# Task 1: Baseline — how wrong is the legacy syllables/4.5 predictor?
# Compares _estimate_duration_baseline(text) vs the real Chatterbox-CPU
# duration recorded in every .align.json sidecar on disk. The mean
# absolute error printed below is the number Task 1 needs to beat.

import statistics
from foreign_whispers.alignment import _estimate_duration_baseline

tts_root = PROJECT_ROOT / "pipeline_data" / "api" / "tts_audio" / "chatterbox"
pairs = []
for align_file in sorted(tts_root.rglob("*.align.json")):
    report = json.loads(align_file.read_text())
    for seg in report.get("segments", []):
        text = seg.get("text", "").strip()
        raw_dur = seg.get("raw_duration_s", 0.0)
        if text and raw_dur > 0:
            pairs.append((text, raw_dur))

if not pairs:
    print("No (text, raw_duration_s) data — run the TTS pipeline first (tts_integration notebook)")
else:
    errs = [abs(_estimate_duration_baseline(t) - dur) for t, dur in pairs]
    print(f"Training corpus: {len(pairs)} segment(s)\n")
    print(f"  Baseline (syllables/4.5)  MAE: {statistics.mean(errs):.3f}s")
    print(f"  Baseline median abs err:        {statistics.median(errs):.3f}s")
    print(f"  Worst single segment:           {max(errs):.3f}s")
    print(f"\nThis is the baseline to beat — see the next cell for the fitted predictor.")


In [ ]:
# Task 1 (continued) — measure the IMPROVED predictor against the baseline.
# After replacing _estimate_duration in foreign_whispers/alignment.py with the
# fitted regression, this cell loads every (text, raw_duration_s) pair from
# every .align.json sidecar on disk, computes MAE for both heuristics, and
# prints the improvement. Re-run after every alignment.py change.

import statistics
from foreign_whispers.alignment import _count_syllables, _estimate_duration

tts_root = PROJECT_ROOT / "pipeline_data" / "api" / "tts_audio"
pairs = []
for align_file in sorted(tts_root.rglob("*.align.json")):
    report = json.loads(align_file.read_text())
    for seg in report.get("segments", []):
        text = seg.get("text", "").strip()
        raw_dur = seg.get("raw_duration_s", 0.0)
        if text and raw_dur > 0:
            pairs.append((text, raw_dur))

if not pairs:
    print("No (text, raw_duration_s) data — run the TTS pipeline first.")
else:
    baseline_errs = [abs(_count_syllables(t) / 4.5 - dur) for t, dur in pairs]
    fitted_errs   = [abs(_estimate_duration(t)   - dur) for t, dur in pairs]

    print(f"Training corpus: {len(pairs)} segment(s)\n")
    print(f"  Baseline (syllables/4.5)  MAE: {statistics.mean(baseline_errs):.3f}s "
          f"  median: {statistics.median(baseline_errs):.3f}s")
    print(f"  Fitted predictor          MAE: {statistics.mean(fitted_errs):.3f}s "
          f"  median: {statistics.median(fitted_errs):.3f}s")

    drop = statistics.mean(baseline_errs) - statistics.mean(fitted_errs)
    pct = 100 * drop / statistics.mean(baseline_errs) if baseline_errs else 0
    print(f"\n  Improvement: {drop:+.3f}s  ({pct:+.0f}% of baseline error)")


## Alignment Fallback Policy

| Stretch Factor | Action            | Description                                |
|----------------|-------------------|--------------------------------------------|
| <= 1.1         | ACCEPT            | Fits naturally, no change needed           |
| 1.1 - 1.4     | MILD_STRETCH      | Apply pyrubberband time-stretch            |
| 1.4 - 1.8     | GAP_SHIFT         | Borrow from adjacent silence gap           |
| 1.8 - 2.5     | REQUEST_SHORTER   | Request a shorter translation              |
| > 2.5         | FAIL              | Unfixable, fall back to silence            |

In [ ]:
action_counts = {a: 0 for a in AlignAction}
for m in all_metrics:
    action_counts[decide_action(m)] += 1

print("Policy distribution:")
for action, count in action_counts.items():
    bar = "\u2588" * count
    print(f"  {action.value:<20} {count:3d}  {bar}")

---

## Task 2: Duration-Aware Translation Re-ranking

Look at the histogram above. Every segment tagged `REQUEST_SHORTER` or `FAIL` is a segment where the Spanish translation is too long to speak in the available time window. The translator doesn't know about duration — it just rewrites text.

**Goal:** For segments that exceed the timing budget, generate shorter translation candidates and pick the one that best fits the source window while preserving meaning.

**Approach:**
1. Filter `all_metrics` for segments where `decide_action(m)` returns `REQUEST_SHORTER`
2. For each, generate 2–3 shorter Spanish alternatives (options: rule-based truncation, LLM candidate generation, or back-translation filtering)
3. Score candidates by: `(predicted_duration - target_duration)² + λ * semantic_distance`
4. Implement this in `foreign_whispers/reranking.py` — the `get_shorter_translations()` stub

**File to modify:** `foreign_whispers/reranking.py`

**Evaluation:**
- How many `REQUEST_SHORTER` segments can you bring down to `MILD_STRETCH` or `ACCEPT`?
- Semantic preservation: compare original and shortened translations using embedding cosine similarity
- Re-run the policy histogram above with your improved translations to measure the shift

### Task 2 — Wire `get_shorter_translations` into the loop

**Approach chosen.** Notebook-only wiring. For every over-budget
segment, call the existing `get_shorter_translations()` from
`foreign_whispers.reranking`, pick the shortest candidate that fits the
~15 chars/sec target, substitute it into a copy of `es_transcript`,
recompute metrics, then print the action-distribution delta.

**Why.** `get_shorter_translations()` is already a complete 3-stage
hybrid (learned cache → rule-based contractions → OpenRouter LLM
fallback) implemented in earlier project work. Re-implementing it in
this notebook would be duplication. The notebook's job here is to
*demonstrate* that calling the function actually moves segments out of
the `REQUEST_SHORTER` / `FAIL` buckets.

**Changes in `foreign_whispers/reranking.py`**: none — the function was complete from prior work.


In [ ]:
# Task 2: Identify the segments that need shorter translations
# These are your targets for re-ranking

over_budget = [m for m in all_metrics if decide_action(m) in (AlignAction.REQUEST_SHORTER, AlignAction.FAIL)]

print(f"Segments needing shorter translations: {len(over_budget)}")
print(f"\nExamples (worst 3):")
for m in sorted(over_budget, key=lambda x: -x.predicted_stretch)[:3]:
    source_dur = m.source_duration_s
    predicted_tts = m.predicted_tts_s
    print(f"\n  seg {m.index}  source_window={source_dur:.1f}s  predicted_tts={predicted_tts:.1f}s  stretch={m.predicted_stretch:.2f}x")
    print(f"    EN: {m.source_text[:70]}")
    print(f"    ES: {m.translated_text[:70]}")
    print(f"    Target: fit TTS into {source_dur:.1f}s → need ~{int(source_dur * 15)} chars or fewer")

In [ ]:
# Task 2: Wire get_shorter_translations into the loop and measure how many
# REQUEST_SHORTER segments resolve. The function is a 3-stage hybrid (learned
# cache → rule-based contractions → OpenRouter LLM); the LLM stage runs only
# when OPENROUTER_API_KEY is set, but rule-based shortening always runs.

from collections import Counter
from foreign_whispers import get_shorter_translations

# Pair each over-budget metric with its source EN text (already on m).
attempts = []
for m in over_budget:
    candidates = get_shorter_translations(
        source_text     = m.source_text,
        baseline_es     = m.translated_text,
        target_duration_s = m.source_duration_s,
    )
    # Pick the shortest candidate that fits our timing budget.
    target_chars = int(m.source_duration_s * 15)  # ~15 chars/sec for Spanish
    fitting = [c for c in candidates if len(c.text) <= target_chars]
    chosen = min(fitting, key=lambda c: len(c.text)) if fitting else None
    attempts.append((m, chosen))

# Build an updated translation by substituting in the chosen shorter text
# for every segment we have a candidate for.
es_segments_v2 = [dict(s) for s in es_transcript["segments"]]
for m, chosen in attempts:
    if chosen is not None:
        es_segments_v2[m.index]["text"] = chosen.text

es_transcript_v2 = {**es_transcript, "segments": es_segments_v2}

# Recompute everything with the shorter translations.
metrics_v2 = compute_segment_metrics(en_transcript, es_transcript_v2)

# Action distribution before vs after.
def _action_counts(metrics_list):
    return Counter(decide_action(m).value for m in metrics_list)

before = _action_counts(all_metrics)
after  = _action_counts(metrics_v2)

resolved = sum(1 for _, c in attempts if c is not None)
print(f"Re-ranking attempted on {len(attempts)} over-budget segment(s)")
print(f"  Resolved (found a fitting candidate): {resolved}")
print(f"  Unresolved (no candidate fit budget): {len(attempts) - resolved}\n")

print(f"  {'action':<18s} {'before':>8s} {'after':>8s}  delta")
for action in [a.value for a in AlignAction]:
    b, a = before.get(action, 0), after.get(action, 0)
    print(f"  {action:<18s} {b:>8d} {a:>8d}  {a-b:+d}")


## Global Timeline Alignment

Optimizer that shifts segments into available silence gaps instead of forcing local stretches.
Uses a greedy left-to-right pass tracking cumulative drift from gap shifts.

In [ ]:
from foreign_whispers import global_align

silence_regions = []  # Would come from VAD if silero-vad installed
aligned_segments = global_align(all_metrics, silence_regions)

shifts = [s for s in aligned_segments if s.action == AlignAction.GAP_SHIFT]
stretches = [s for s in aligned_segments if s.action == AlignAction.MILD_STRETCH]
drift = aligned_segments[-1].scheduled_end - aligned_segments[-1].original_end if aligned_segments else 0.0

print(f"Gap shifts     : {len(shifts)}")
print(f"Mild stretches : {len(stretches)}")
print(f"Total drift    : {drift:.2f}s")

## Visualize Alignment Timeline

Plot original vs scheduled timing for each segment.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

y_original = 1
y_scheduled = 0

for seg in aligned_segments:
    # Original timing (blue)
    ax.barh(y_original, seg.original_end - seg.original_start,
            left=seg.original_start, height=0.3, color="steelblue", alpha=0.6,
            edgecolor="none")
    # Scheduled timing (orange)
    ax.barh(y_scheduled, seg.scheduled_end - seg.scheduled_start,
            left=seg.scheduled_start, height=0.3, color="darkorange", alpha=0.6,
            edgecolor="none")

ax.set_yticks([y_scheduled, y_original])
ax.set_yticklabels(["Scheduled (target)", "Original (source)"])
ax.set_xlabel("Time (seconds)")
ax.set_title("Original vs Scheduled Segment Timing")
ax.set_ylim(-0.5, 1.8)
fig.tight_layout()
fig.savefig(IMAGES_DIR / "alignment_timeline.png", dpi=150)
plt.show()
print(f"Saved to {IMAGES_DIR / 'alignment_timeline.png'}")

---

## Task 3: Beat the Greedy Optimizer

The timeline above was produced by `global_align()` — a greedy left-to-right pass. It makes locally optimal decisions but can't look ahead. A segment that borrows silence early may starve a later segment that needed it more.

**Goal:** Implement a better global optimizer and compare it against the greedy baseline.

**Approach:**
1. Record the greedy baseline metrics: total drift, number of gap shifts, number of segments still in `REQUEST_SHORTER` or `FAIL` after alignment
2. Implement one of these alternatives in a new function (e.g. `global_align_dp` in `foreign_whispers/alignment.py`):
   - **Dynamic programming:** minimize total stretch penalty over all segments, subject to non-overlapping constraints
   - **Integer linear programming:** formulate as an optimization problem with scipy or PuLP — decision variables are per-segment time allocations, constraints enforce non-overlap and silence budgets
   - **Beam search:** explore multiple scheduling trajectories, prune by cumulative drift
3. Compare your optimizer against the greedy baseline on the same clip

**File to modify:** `foreign_whispers/alignment.py` — add your optimizer alongside `global_align`

**Evaluation:**
- Total cumulative drift (lower is better)
- Number of segments requiring severe stretch (>1.4x)
- Number of segments that overlap in the scheduled timeline
- Re-plot the timeline visualization above with your optimizer's output

### Task 3 — Beat the greedy optimiser

**Approach chosen.** Forward dynamic programming over the state
`(segment_index, cumulative_drift_quantum)` with a weighted cost
function `overflow + severe_stretch_indicator + drift²`. Cost weights
and the drift quantisation are kwargs so this notebook can sweep them.
The value function is LRU-cached; the schedule is reconstructed
forward by matching `local_cost + V(i+1, drift')` against `V(i, drift)`.

**Why.** Greedy `decide_action` picks one action per segment based on
local thresholds; it can never trade "spend gap on segment 5" against
"save it for segment 10". DP enumerates *every* greedy candidate plus
alternatives (e.g. MILD_STRETCH-at-cap when GAP_SHIFT is also
feasible), so DP's candidate set is a strict superset of greedy's —
DP cost is provably ≤ greedy cost. The **drift² term is what makes DP
non-trivially better**: borrowing gap *now* enlarges every future
segment's drift state, so the optimiser learns to skip a gap-shift
when the local overflow penalty is cheaper than carrying drift through
the rest of the timeline. Quantising drift to 0.1 s keeps state size
manageable (O(N · 100) for clips up to a few minutes).

**Changes in `foreign_whispers/alignment.py`**:

- New function `global_align_dp()` (~110 lines) with internal helpers `_silence_after`, `_candidates`, `_value`.

**Changes in `foreign_whispers/__init__.py`**: re-exported `global_align_dp` and added it to `__all__`.


In [ ]:
# Task 3: Beat the greedy optimizer.
#
# global_align_dp is a forward DP over (segment_index, cumulative_drift_quantum)
# that enumerates every feasible action per segment and minimises a weighted
# cost: overflow + severe-stretch indicator + drift². Every action greedy
# considers is also a DP candidate, so DP's schedule is optimal under that
# cost function and weakly dominates greedy.

from foreign_whispers import clip_evaluation_report, global_align_dp

dp_segments = global_align_dp(all_metrics, silence_regions=silence_regions)

greedy_report = clip_evaluation_report(all_metrics, aligned_segments)
dp_report     = clip_evaluation_report(all_metrics, dp_segments)

print("=== Greedy baseline ===")
for k, v in greedy_report.items():
    print(f"  {k:30s} {v}")

print("\n=== DP optimiser ===")
for k, v in dp_report.items():
    delta = v - greedy_report[k] if isinstance(v, (int, float)) else None
    if isinstance(delta, (int, float)):
        marker = "↓" if delta < 0 else ("↑" if delta > 0 else "=")
        print(f"  {k:30s} {v}   ({marker} {delta:+.3f} vs greedy)")
    else:
        print(f"  {k:30s} {v}")

# Action distribution side-by-side
from collections import Counter
g_dist = Counter(s.action.value for s in aligned_segments)
d_dist = Counter(s.action.value for s in dp_segments)
print("\n=== Action distribution ===")
print(f"  {'action':<18s} {'greedy':>8s} {'dp':>8s}")
for action in [a.value for a in AlignAction]:
    print(f"  {action:<18s} {g_dist.get(action, 0):>8d} {d_dist.get(action, 0):>8d}")


In [ ]:
# Side-by-side timeline visualisation: greedy (top) vs DP (bottom).
import matplotlib.pyplot as plt

fig, (ax_g, ax_d) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

ACTION_COLORS = {
    AlignAction.ACCEPT:          "#5fbf5f",
    AlignAction.MILD_STRETCH:    "#f0c850",
    AlignAction.GAP_SHIFT:       "#5099d8",
    AlignAction.REQUEST_SHORTER: "#d87050",
    AlignAction.FAIL:            "#a04050",
}

def _draw(ax, segments, label):
    for seg in segments:
        ax.barh(
            0,
            seg.scheduled_end - seg.scheduled_start,
            left=seg.scheduled_start,
            height=0.6,
            color=ACTION_COLORS.get(seg.action, "#888"),
            edgecolor="#333",
            linewidth=0.4,
        )
    ax.set_yticks([])
    ax.set_ylabel(label, fontsize=11)
    ax.set_xlim(0, max(s.scheduled_end for s in segments) * 1.02)

_draw(ax_g, aligned_segments, "greedy")
_draw(ax_d, dp_segments,      "dp")
ax_d.set_xlabel("scheduled time (s)")
ax_d.legend(
    handles=[plt.Rectangle((0,0),1,1,color=c,label=a.value) for a,c in ACTION_COLORS.items()],
    loc="upper right", ncol=5, fontsize=8,
)
plt.tight_layout()
plt.savefig(IMAGES_DIR / "greedy_vs_dp_timeline.png", dpi=110, bbox_inches="tight")
plt.show()
print(f"Saved → {IMAGES_DIR / 'greedy_vs_dp_timeline.png'}")


---

## Task 4: Build a Dubbing Quality Scorecard

The `clip_evaluation_report()` above gives you five numbers. But dubbing quality is multi-dimensional — timing accuracy is necessary but not sufficient. A clip with perfect timing but garbled speech is still a failure.

**Goal:** Design and implement a richer evaluation framework that scores clips across multiple dimensions.

**Dimensions to consider:**
- **Timing accuracy:** mean absolute duration error, percentage of severe stretches, cumulative drift (you already have these)
- **Intelligibility:** can you use a speech-to-text round-trip? TTS the Spanish, then STT it back — compare against the translation. Word error rate of the round-trip measures intelligibility.
- **Semantic fidelity:** how much meaning was lost? Compare source English and back-translated English using embedding cosine similarity
- **Naturalness:** speaking rate variance across segments — is it consistent or does it jump between fast and slow?

**Approach:**
1. Implement `dubbing_scorecard(metrics, aligned_segments, align_report)` in `foreign_whispers/evaluation.py`
2. Return a dict with scores per dimension, each normalized to [0, 1]
3. Add a summary visualization — a radar chart or bar chart comparing baseline vs aligned

**File to modify:** `foreign_whispers/evaluation.py`

**Evaluation:**
- Does your scorecard distinguish between good and bad clips?
- Do the dimensions correlate with each other, or do they capture independent quality aspects?
- Run on multiple videos from `video_registry.yml` and compare

### Task 4 — Multi-dimensional dubbing scorecard

**Approach chosen.** A scorecard returning sub-scores in `[0, 1]` for
*timing*, *naturalness*, *intelligibility*, and *semantic fidelity*,
plus an `overall_score` that's the unweighted mean of whichever
dimensions actually got computed. Heavy services (Whisper STT, ES→EN
back-translator, sentence embedder) are injected as
**`typing.Protocol`** classes — when a service is absent the
corresponding dimension is silently dropped.

**Why.** Timing alone is insufficient — a clip with perfect timing but
unintelligible speech is still a failure. Protocol injection lets the
function work in three modes: (1) timing+naturalness only (cheap, no
services needed), (2) +intelligibility (with Whisper), (3) +semantic
(with back-translator + embedder). Tests pin behaviour in all three
modes via mocks, so we don't need to load Whisper or download model
weights to verify the score logic. Word-level Levenshtein is the
research-standard ASR evaluation; `all-MiniLM-L6-v2` (22 MB) is the
standard "semantic similarity on a budget" embedder; cosine similarity
uses stdlib `math` so the function has no hard `numpy` dependency.

The timing dimension penalises only *real* problems (severe stretches
+ cumulative drift) — not `mean_abs_duration_error_s`, which stays
high even on perfect "play at natural speed + pad with silence" plans.

**Changes in `foreign_whispers/evaluation.py`**:

- Protocol classes `WhisperService / BackTranslator / Embedder`.
- Helpers `_word_error_rate()` (DP edit distance) and `_cosine_similarity()` (stdlib only).
- Main function `dubbing_scorecard()`.

**Changes in `foreign_whispers/__init__.py`**: re-exported `dubbing_scorecard`.

**Changes in `pyproject.toml`**: added `sentence-transformers>=2.2` to main dependencies (model downloads on first encode call).


In [ ]:
# Task 4: dubbing_scorecard — multi-dimensional evaluation.
#
# Heavy services (Whisper STT round-trip, argostranslate back-translator,
# sentence-transformers embedder) are injected as Protocols. Run order:
#   1. Build per-config Whisper / argostranslate / embedder adapters.
#   2. Score the greedy-aligned config and the dp-aligned config.
#   3. Side-by-side bar chart so Task 3's improvements are visible.

import os, requests, pathlib
from foreign_whispers import dubbing_scorecard

# ── Adapters ────────────────────────────────────────────────────────
WHISPER_URL = os.getenv("FW_WHISPER_API_URL", "http://localhost:8000")

class WhisperHTTPAdapter:
    """Calls the running Whisper container at /v1/audio/transcriptions."""
    def transcribe(self, wav_path: str, language: str = "es") -> str:
        with open(wav_path, "rb") as f:
            r = requests.post(
                f"{WHISPER_URL}/v1/audio/transcriptions",
                files={"file": (pathlib.Path(wav_path).name, f, "audio/wav")},
                data={"language": language},
                timeout=(5, 600),
            )
        r.raise_for_status()
        body = r.json()
        return body.get("text", "") if isinstance(body, dict) else str(body)

class ArgosBackTranslator:
    """ES → EN via argostranslate (assumes the package is installed)."""
    def translate(self, text: str, src: str = "es", dst: str = "en") -> str:
        import argostranslate.translate as at
        return at.translate(text, src, dst)

class STEmbedder:
    """sentence-transformers all-MiniLM-L6-v2 — small + CPU-friendly."""
    _model = None
    def encode(self, texts):
        if STEmbedder._model is None:
            from sentence_transformers import SentenceTransformer
            STEmbedder._model = SentenceTransformer("all-MiniLM-L6-v2")
        return STEmbedder._model.encode(texts, convert_to_numpy=True).tolist()

# ── Score both schedulers on the cached audio ────────────────────────
title = "Strait of Hormuz disruption threatens to shake global economy"
audio_path = str(PROJECT_ROOT / "pipeline_data/api/tts_audio/chatterbox/c-fb1074a" / f"{title}.wav")
audio_exists = pathlib.Path(audio_path).is_file()

if not audio_exists:
    print(f"!! No TTS audio at {audio_path} — intelligibility will be skipped.")

whisper_svc = WhisperHTTPAdapter() if audio_exists else None
back_tx     = ArgosBackTranslator()
embedder    = STEmbedder()

def _score(metrics_, aligned_, label):
    report = clip_evaluation_report(metrics_, aligned_)
    sc = dubbing_scorecard(
        metrics_, aligned_, report,
        audio_path=audio_path if audio_exists else None,
        whisper=whisper_svc,
        back_translator=back_tx,
        embedder=embedder,
    )
    print(f"\n=== {label} ===")
    for k, v in sc.items():
        print(f"  {k:24s} {v}")
    return sc

greedy_sc = _score(all_metrics, aligned_segments, "greedy")
dp_sc     = _score(all_metrics, dp_segments,      "dp")


In [ ]:
# Side-by-side bar chart of the two scorecards.
import matplotlib.pyplot as plt
import numpy as np

dims = [k for k in greedy_sc.keys() if k != "overall_score"]
greedy_vals = [greedy_sc[k] for k in dims]
dp_vals     = [dp_sc[k]     for k in dims]

x = np.arange(len(dims))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width/2, greedy_vals, width, label="greedy", color="#888")
ax.bar(x + width/2, dp_vals,     width, label="dp",     color="#5099d8")
ax.set_xticks(x)
ax.set_xticklabels([d.replace("_score", "") for d in dims], rotation=15)
ax.set_ylim(0, 1.05)
ax.set_ylabel("score (1 = best)")
ax.set_title(f"Dubbing Scorecard — overall: greedy={greedy_sc['overall_score']}  dp={dp_sc['overall_score']}")
ax.legend()
ax.grid(axis="y", linestyle=":", alpha=0.5)
plt.tight_layout()
plt.savefig(IMAGES_DIR / "scorecard_greedy_vs_dp.png", dpi=110, bbox_inches="tight")
plt.show()
print(f"Saved → {IMAGES_DIR / 'scorecard_greedy_vs_dp.png'}")


---

## Summary

Alignment is pure Python, no GPU required. All the timing analysis, policy decisions,
and global scheduling run on CPU with zero external dependencies beyond stdlib.

### Task overview

| Task | What you build | File to modify | Evaluation |
|------|---------------|----------------|------------|
| 1. Duration Prediction | Better TTS duration estimator | `alignment.py` — `_estimate_duration` | Mean absolute error vs ground truth |
| 2. Translation Re-ranking | Shorter candidates that fit the timing budget | `reranking.py` — `get_shorter_translations` | Segments moved from `REQUEST_SHORTER` to `ACCEPT` |
| 3. Global Optimizer | DP/ILP/search optimizer that beats greedy | `alignment.py` — new `global_align_dp` | Total drift, severe stretch count |
| 4. Quality Scorecard | Multi-dimensional evaluation framework | `evaluation.py` — new `dubbing_scorecard` | Dimension independence, cross-clip consistency |

Each task builds on the analysis you ran above and uses data already in `pipeline_data/`.
Validate your results end-to-end by re-running the TTS and stitch notebooks.

### Cross-notebook connections

- **Speaker-aware alignment** — the `diarization_integration` notebook labels speaker turns.
  Feed that signal into your optimizer to prevent borrowing silence across speaker boundaries.
- **Voice cloning** — the `tts_integration` notebook wires per-speaker voice selection.
  Combined with diarization, different speakers get different voices.
